In [1]:
import pandas as pd

# Import Data

Data source: EM-DAT, CRED (https://www.emdat.be/)

In [2]:
df = pd.read_excel("public_emdat_incl_hist_2026-02-28.xlsx", sheet_name="EM-DAT Data")

# columns to keep from original dataset
keep = [
    "DisNo.", "Disaster Group", "Disaster Type", "Disaster Subtype",
    "ISO", "Country", "Region",
    "Latitude", "Longitude",
    "Start Year", "Start Month", "Start Day",
    "End Year", "End Month", "End Day",
    "Total Deaths", "Total Affected"
]

df = df.loc[:, keep]


# Filter and Clean

Keeping only `Natural` disasters (~65%); all remaining disasters are classified as `Technological` (~35%).

In [3]:
# standardise column names
df.columns = list(map(lambda col: col.lower().replace(" ", "_"), df.columns.tolist()))
df = df.rename(columns={"DisNo.": "disaster_id"})       # rename unique disaster identifier

# keep natural disasters only (~65% of available data)
df = df.loc[df["disaster_group"] == "Natural"]

# convert available start date data to datetime
df["start_datetime"] = pd.to_datetime(
    df[["start_year", "start_month", "start_day"]]      # pass DataFrame (YMD columns) as input
    .rename(
        columns={
            "start_year": "year",       # method expects exact naming
            "start_month": "month",
            "start_day": "day"
        }
    )
)

# convert available end date data to datetime
df["end_datetime"] = pd.to_datetime(
    df[["end_year", "end_month", "end_day"]]      # pass DataFrame (YMD columns) as input
    .rename(
        columns={
            "end_year": "year",       # method expects exact naming
            "end_month": "month",
            "end_day": "day"
        }
    )
)

# calculate disaster duration in days
df["duration_days"] = (df["end_datetime"] - df["start_datetime"]).dt.days.fillna(0)

# drop day and month data after creating datetimes
df = df.drop(columns=[
    "start_month", "start_day",
    "end_month", "end_day", "disaster_group"
])

# Export to CSV

In [4]:
df.to_csv("natural_disasters.csv", index=False)

In [5]:
df = pd.read_csv("natural_disasters.csv", parse_dates=["start_datetime", "end_datetime"])

df.head()

,disno.,disaster_type,disaster_subtype,iso,country,region,latitude,longitude,start_year,end_year,total_deaths,total_affected,start_datetime,end_datetime,duration_days
0,1900-0003-USA,Storm,Tropical cyclone,USA,United States of America,Americas,NaN,NaN,1900,1900,6000.0,NaN,1900-09-08,1900-09-08,0.0
1,1900-0006-JAM,Flood,Flood (General),JAM,Jamaica,Americas,NaN,NaN,1900,1900,300.0,NaN,1900-01-06,1900-01-06,0.0
2,1900-0007-JAM,Epidemic,Viral disease,JAM,Jamaica,Americas,NaN,NaN,1900,1900,30.0,NaN,1900-01-13,1900-01-13,0.0
3,1900-0008-JPN,Volcanic activity,Ash fall,JPN,Japan,Asia,NaN,NaN,1900,1900,30.0,NaN,1900-07-07,1900-07-07,0.0
4,1900-0009-TUR,Earthquake,Ground movement,TUR,Türkiye,Asia,40.3,43.1,1900,1900,140.0,NaN,1900-07-12,1900-07-12,0.0
